In [1]:
import pandas as pd
#import xlrd
import openpyxl
#import defusedxml

# Function to parse the lower bound duration from FD duration range
def parse_duration(range_str):
    try:
        range_str = range_str.strip().lower()  # Normalize the input text
        #print("duration string = ",range_str)

        # Handle specific cases like "Tax-saver Fixed Deposit"
        if "tax-saver" in range_str:
            return 5 * 365  # Tax-saver Fixed Deposit = 5 years = 1825 days
        
        # Handle cases like ">1 Years - 443 Days"
        if '>' in range_str:
            for delimiter in ['-', 'to']:
                if delimiter in range_str:
                    lower_bound = range_str.split(delimiter)[0].strip()
                    #print("lower bound in > =",lower_bound)
                    break
            days=0
            #lower_bound = range_str.split('-')[0].strip()
            if any(x in lower_bound for x in ['year', 'yr']):
                for delimiter in ['year', 'yr']:
                    if delimiter in lower_bound:
                        parts = lower_bound.split(delimiter)
                        break
                years = int(''.join(filter(str.isdigit, parts[0])))  # Extract years
                days += years * 365  # Convert years to days
                #print("days in year =",days)
                #print("parts1 =",parts[1] )
                if 'month' in parts[1]:
                    month_part = parts[1].split('month')
                    months = int(''.join(filter(str.isdigit, month_part[0])))  # Extract months
                    days += months * 30  # Approximate: Convert months to days
                    parts[1] = month_part[1]
                    #print("days in month =",days)
                    #print("months part 1=",month_part[1])
                if 'day' in parts[1]:
                    day_part = parts[1].split('day')
                    #print("day_part=",day_part)
                    days += int(''.join(filter(str.isdigit, day_part[0])))  # Extract days
                #print("days after =",days)
                return days
            elif any(x in lower_bound for x in ['month', 'mth']):
                #print("lower bound in month")
                for delimiter in ['month', 'mth']:
                    if delimiter in range_str:
                        parts = lower_bound.split(delimiter)
                        #print("parts =",parts)
                        break
                months = int(''.join(filter(str.isdigit, parts[0])))  # Extract years
                days += months * 30  # Convert years to days
                if 'day' in parts[1]:
                    days += int(''.join(filter(str.isdigit, parts[1])))  # Add days
                return days
            elif "day" in lower_bound:
                day_part = lower_bound.split('day')
                #print("day_part=",day_part)
                days += int(''.join(filter(str.isdigit, day_part[0])))  # Extract days
                return days
        
        
        '''
        # Handle cases like ">1 Years - 443 Days"
        if '>' in range_str:
            for delimiter in ['-', 'to']:
                if delimiter in range_str:
                    lower_bound = range_str.split(delimiter)[0].strip()
                    print("lower bound in > =",lower_bound)
                    break
            #lower_bound = range_str.split('-')[0].strip()
            if any(x in lower_bound for x in ['year', 'yr']):
                years = int(''.join(filter(str.isdigit, lower_bound)))
                return years * 365  # Convert years to days
            elif any(x in lower_bound for x in ['month', 'mth']):
                months = int(''.join(filter(str.isdigit, lower_bound)))
                return months * 30  # Convert months to days
            elif 'day' in lower_bound:
                days = int(''.join(filter(str.isdigit, lower_bound)))
                return days # Extract numeric days
        '''
        
        # Handle cases like "1 year 1 day to 1 year 9 months"
        if 'to' in range_str:
            lower_bound = range_str.split('to')[0].strip()
            print("lower_bound in to =",lower_bound)
            days = 0
            if any(x in lower_bound for x in ['year', 'yr']):
                for delimiter in ['year', 'yr']:
                    if delimiter in lower_bound:
                        parts = lower_bound.split(delimiter)
                        break
                years = int(''.join(filter(str.isdigit, parts[0])))  # Extract years
                days += years * 365  # Convert years to days
                #print("days=",days)
                #print("parts1 =",parts[1] )
                if 'month' in parts[1]:
                    month_part = parts[1].split('month')
                    months = int(''.join(filter(str.isdigit, month_part[0])))  # Extract months
                    days += months * 30  # Approximate: Convert months to days
                    parts[1] = month_part[1]
                    #print("days in month =",days)
                    #print("months part 1=",month_part[1])
                if 'day' in parts[1]:
                    day_part = parts[1].split('day')
                    #print("day_part=",day_part)
                    days += int(''.join(filter(str.isdigit, day_part[0])))  # Extract days
                    
                '''
                if 'day' in parts[1]:
                    days += int(''.join(filter(str.isdigit, parts[1])))  # Add days
                    print("days in day=",days)
                elif 'month' in parts[1]:
                    months = int(''.join(filter(str.isdigit, parts[1])))  # Extract months
                    days += months * 30  # Approximate: Convert months to days
                '''
                #print("days after =",days)
                return days
            elif any(x in lower_bound for x in ['month', 'mth']):
                #print("lower bound in month")
                for delimiter in ['month', 'mth']:
                    if delimiter in range_str:
                        parts = lower_bound.split(delimiter)
                        #print("parts =",parts)
                        break
                months = int(''.join(filter(str.isdigit, parts[0])))  # Extract years
                days += months * 30  # Convert years to days
                if 'day' in parts[1]:
                    days += int(''.join(filter(str.isdigit, parts[1])))  # Add days
                return days
            elif "day" in lower_bound:
                day_part = lower_bound.split('day')
                #print("day_part=",day_part)
                days += int(''.join(filter(str.isdigit, day_part[0])))  # Extract days
                return days
        
        if '-' in range_str or '–' in range_str or '<' in range_str or 'less' in range_str:
            lower_bound = range_str.split('-')[0].split('–')[0].split('<')[0].split('less')[0].strip()
            #print("lower_bound in - =",lower_bound)
            days = 0
            if any(x in lower_bound for x in ['year', 'yr']):
                for delimiter in ['year', 'yr']:
                    if delimiter in lower_bound:
                        parts = lower_bound.split(delimiter)
                        break
                years = int(''.join(filter(str.isdigit, parts[0])))  # Extract years
                days += years * 365  # Convert years to days
                #print("days=",days)
                #print("parts1 =",parts[1] )
                if 'month' in parts[1]:
                    month_part = parts[1].split('month')
                    months = int(''.join(filter(str.isdigit, month_part[0])))  # Extract months
                    days += months * 30  # Approximate: Convert months to days
                    parts[1] = month_part[1]
                    #print("days in month =",days)
                    #print("months part 1=",month_part[1])
                if 'day' in parts[1]:
                    day_part = parts[1].split('day')
                    #print("day_part=",day_part)
                    days += int(''.join(filter(str.isdigit, day_part[0])))  # Extract days
                #print("days after =",days)
                return days
            elif any(x in lower_bound for x in ['month', 'mth']):
                #print("lower bound in month")
                for delimiter in ['month', 'mth']:
                    if delimiter in lower_bound:
                        parts = lower_bound.split(delimiter)
                        #print("parts =",parts)
                        break
                months = int(''.join(filter(str.isdigit, parts[0])))  # Extract years
                days += months * 30  # Convert years to days
                if 'day' in parts[1]:
                    days += int(''.join(filter(str.isdigit, parts[1])))  # Add days
                return days
            elif "day" in lower_bound:
                #print("lower bound in day =",lower_bound)
                days += int(''.join(filter(str.isdigit, lower_bound)))  # Extract and return only the lower bound
                return days
            else:
                if "day" in range_str:
                    return int(''.join(filter(str.isdigit, lower_bound)))  # Extract numeric days
                elif any(x in range_str for x in ['month', 'mth']):
                    return int(''.join(filter(str.isdigit, lower_bound))) * 30  # Convert months to days
                elif any(x in range_str for x in ['year', 'yr']):
                    return int(''.join(filter(str.isdigit, lower_bound))) * 365  # Convert years to days
                else:
                    return int(''.join(filter(str.isdigit, lower_bound)))  # Extract numeric days
            #return days
                 
        # Handle "to less than" cases like "to less than 1 year"
        if "to" in range_str:
            lower_bound = range_str.split("to")[0].strip()
            if "day" in lower_bound:
                return int(''.join(filter(str.isdigit, lower_bound)))  # Extract numeric days
            elif any(x in lower_bound for x in ['month', 'mth']):
                return int(''.join(filter(str.isdigit, lower_bound))) * 30  # Convert months to days
            elif any(x in lower_bound for x in ['year', 'yr']):
                return int(''.join(filter(str.isdigit, lower_bound))) * 365  # Convert years to days
            else:
                return int(lower_bound)
        
        # Handle year-based entries
        if "year" in range_str:
            #print("in years")
            years = int(''.join(filter(str.isdigit, range_str.split("year")[0])))
            return years * 365  # Approximate: Convert years to days
        
        # Handle month-based entries like "22 Months (PSB Green Earth)"
        if "month" in range_str:
            months = int(''.join(filter(str.isdigit, range_str.split("month")[0])))
            return months * 30  # Approximate: Convert months to days
                      
        # Handle explicit values like "444 days" or "1 yr"
        if "day" in range_str:
            return int(''.join(filter(str.isdigit, range_str.split("day")[0])))  # Extract numeric days
        elif any(x in range_str for x in ['year', 'yr']):
            return int(''.join(filter(str.isdigit, range_str))) * 365  # Convert years to days
                
        return None  # Default for unhandled cases
    except Exception as e:
        print("Error parsing duration: {}. Exception: {}".format(range_str, e))
        return None

# Function to compute annualized yield or retain nominal rate
def compute_yield_or_nominal(row, rate_column):
    """
    Computes the annualized yield or retains the nominal rate based on the duration.

    Parameters:
        row: A row of the DataFrrate_column: The column name containing the nominal interest rate.

    Returns:
        Annualized yield (percentage) if duration >= 180 days; otherwise, the nominal rate (percentage).
    """
    nominal_rate = row[rate_column]  # Convert rate to a decimal
    duration_days = row['Duration (days)']  # Get the duration in days

    if duration_days >= 180:
        # Compute the annualized yield using quarterly compounding
        n = 4  # Quarterly compounding
        T = duration_days / 365.0  # Convert days to years
        effective_yield = (1 + (nominal_rate / n)) ** (n * T) - 1
        print("nominal rate = ", nominal_rate)
        print("effective yield = ",effective_yield)
        annualized_yield = (1 + effective_yield) ** (1 / T) - 1
        print("annualised yield = ", annualized_yield)
        print("duration = ",duration_days)
        return annualized_yield  # Convert back to percentage
    else:
        # Retain the nominal rate for durations < 180 days
        return row[rate_column]  # Return as percentage
    '''
    if duration_days >= 180:
        # Compute the annualized yield using quarterly compounding
        n = 4  # Quarterly compounding
        T = duration_days / 365.0  # Convert days to years
        annualized_yield = (1 + nominal_rate / n) ** (n * T) - 1
        return annualized_yield * 100  # Convert back to percentage
    else:
        # Retain the nominal rate for durations < 180 days
        return row[rate_column]  # Return as percentage
    '''

#Main program starts here
FILE_PATH = r'c:\Interest_rate_files'

# Load the Excel file
file_name = FILE_PATH + '\Interest_rate_file.xlsx'  # Update with the correct file path
#file_name = FILE_PATH + '\Small_input_table.xlsx'  # Update with the correct file path
interest_table_df = pd.read_excel(file_name, engine='openpyxl')

#pd.set_option('display.max_columns', None)
#print(interest_table_df)

# Parse the FD duration range to extract lower bound duration
interest_table_df['Duration (days)'] = interest_table_df['tenure'].apply(parse_duration)
#output_file_name = FILE_PATH + '\ProcessedTable.csv'
#output_file_name = FILE_PATH + '\Small_output_table.csv'
#interest_table_df.to_csv(output_file_name, mode='w', header=True, index=False)
#print("Processing complete")

# Compute annualized yields for General Rate and Senior Citizen Rate
interest_table_df['Annualised yield (General)'] = interest_table_df.apply(
    lambda row: compute_yield_or_nominal(row, 'gen_rate'), axis=1)
interest_table_df['Annualised yield (Senior Citizen)'] = interest_table_df.apply(
    lambda row: compute_yield_or_nominal(row, 'sr_rate'), axis=1)


# Save the updated table to a new Excel file
output_file_name = FILE_PATH + '\Processed_Interest_Table.xlsx'
interest_table_df.to_excel(output_file_name, index=False)

print("Processed file saved as: {}".format(output_file_name))


lower_bound in to = 7 days
lower_bound in to = 1 month 16 days
lower_bound in to = 3 months 1 day
lower_bound in to = 6 months 1 day
lower_bound in to = 12 months 1 day
lower_bound in to = 15 months 1 day
lower_bound in to = 18 months 1 day
lower_bound in to = 24 months 1 day
lower_bound in to = 36 months 1 day
lower_bound in to = 45 months 1 day
lower_bound in to = 60 months
lower_bound in to = 5 years
lower_bound in to = 7 days
lower_bound in to = 15 days
lower_bound in to = 31 days
lower_bound in to = 2 months
lower_bound in to = 3 months
lower_bound in to = 6 months
lower_bound in to = 1 year 1 day
lower_bound in to = 1 year 9 months 1 day
lower_bound in to = 2 years
lower_bound in to = 3 years
lower_bound in to = 5 years
lower_bound in to = 7 days
lower_bound in to = 15 days
lower_bound in to = 46 days
lower_bound in to = 91 days
lower_bound in to = 181 days
lower_bound in to = 211 days
lower_bound in to = above 1 year
lower_bound in to = above 400 days and up
lower_bound in to = 

Processed file saved as: c:\Interest_rate_files\Processed_Interest_Table.xlsx


In [2]:
!python --version

Python 3.8.20
